This notebook joins all CSU individual datasets with
- name | First Last
- title
- department
- school
- start year | null
- major | null

In [1]:
import pandas as pd
import duckdb
import numpy as np
from pathlib import Path

In [4]:
#read every parquet in ../Data
data_dir = Path("../Data").resolve()
output_path = data_dir / "csu_combined.parquet"

parquet_files = sorted(
    path for path in data_dir.glob("*.parquet")
    if path != output_path
)

if not parquet_files:
    raise FileNotFoundError(f"No input parquet files found in {data_dir}")

combined = pd.concat(
    [pd.read_parquet(path) for path in parquet_files],
    axis=0,
    join="outer",
    ignore_index=True,
    sort=False,
)

combined.to_parquet(output_path, index=False)

print(f"Combined {len(parquet_files)} files: {combined.shape[0]:,} rows, "
      f"{combined.shape[1]} columns")
print(f"Saved to: {output_path}")
print(combined.shape)
combined.head(10)

Combined 21 files: 16,984 rows, 8 columns
Saved to: C:\Users\ginny\Classes-Other\DataEng\CSC4693\Faculty\Data\csu_combined.parquet
(16984, 8)


,name,title,department,school,year,major,source_url,start_date
0,Dawn Abel,Lecturer,Nursing,"California State University, Chico",2013,<NA>,NaN,NaN
1,Annie Adamian,Associate Professor,Education,"California State University, Chico",2014,<NA>,NaN,NaN
2,Nora Aguilar McKay,Lecturer,Education,"California State University, Chico",2014,<NA>,NaN,NaN
3,Kelsey Aguire,Lecturer,Nursing,"California State University, Chico",2025,<NA>,NaN,NaN
4,Kirk Aiken,Professor,Marketing,"California State University, Chico",2013,<NA>,NaN,NaN
5,Hannah Aird,Associate Professor,Earth & Environmental Sci,"California State University, Chico",2015,<NA>,NaN,NaN
6,Oleksiy Al-Saadi,Assistant Professor,Computer Science,"California State University, Chico",2025,<NA>,NaN,NaN
7,Zahrasadat Alavi,Associate Professor,Electrical & Computer Engr,"California State University, Chico",2017,<NA>,NaN,NaN
8,Mohammed Albahttiti,Associate Professor,Concrete Industry Mgmt,"California State University, Chico",2017,<NA>,NaN,NaN
9,Kelli Albertson,Lecturer,Physics; Science Education,"California State University, Chico",2008,<NA>,NaN,NaN


In [7]:
csu_pay = pd.read_parquet('../Data/csu_faculty_2024.parquet')
csu_pay.head(5)

,name,job,department,base,overtime,other,benefitsee,benefitser,benefitsdc,benefitshw,ual,totalpay,totalbenefits,totalpaybenefits,notes,year,agency,status
0,Stephen Charles Sillett,Instructional Faculty - 12 Month,NaN,147768.12,0.0,0.00,NaN,NaN,NaN,NaN,NaN,147768.12,69547.66,217315.78,NaN,2024,California State University,NaN
1,Arne Edward Jacobson,Instructional Faculty - 12 Month,NaN,144585.88,0.0,0.00,NaN,NaN,NaN,NaN,NaN,144585.88,76211.20,220797.08,NaN,2024,California State University,NaN
2,Rouhollah Aghasaleh,Instructional Faculty - Academic Year,NaN,88802.72,0.0,0.00,NaN,NaN,NaN,NaN,NaN,88802.72,55334.46,144137.18,NaN,2024,California State University,NaN
3,Josephine A Archibald,Instructional Faculty - Academic Year,NaN,82308.88,0.0,18542.17,NaN,NaN,NaN,NaN,NaN,100851.05,57649.07,158500.12,NaN,2024,California State University,NaN
4,Joseph Oduro Appiah,Instructional Faculty - Academic Year,NaN,642.00,0.0,3000.00,NaN,NaN,NaN,NaN,NaN,3642.00,205.44,3847.44,NaN,2024,California State University,NaN


In [8]:
csu_pay["name"] = (
    csu_pay["name"]
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
    .str.replace(r"(?<=\S) (?:[A-Za-z]\.? )+(?=\S)", " ", regex=True)
    .str.replace(r"(?i),? (?:Jr\.?|Sr\.?|II|III|IV)$", "", regex=True)
)

csu_pay.head(5)

,name,job,department,base,overtime,other,benefitsee,benefitser,benefitsdc,benefitshw,ual,totalpay,totalbenefits,totalpaybenefits,notes,year,agency,status
0,Stephen Charles Sillett,Instructional Faculty - 12 Month,NaN,147768.12,0.0,0.00,NaN,NaN,NaN,NaN,NaN,147768.12,69547.66,217315.78,NaN,2024,California State University,NaN
1,Arne Edward Jacobson,Instructional Faculty - 12 Month,NaN,144585.88,0.0,0.00,NaN,NaN,NaN,NaN,NaN,144585.88,76211.20,220797.08,NaN,2024,California State University,NaN
2,Rouhollah Aghasaleh,Instructional Faculty - Academic Year,NaN,88802.72,0.0,0.00,NaN,NaN,NaN,NaN,NaN,88802.72,55334.46,144137.18,NaN,2024,California State University,NaN
3,Josephine Archibald,Instructional Faculty - Academic Year,NaN,82308.88,0.0,18542.17,NaN,NaN,NaN,NaN,NaN,100851.05,57649.07,158500.12,NaN,2024,California State University,NaN
4,Joseph Oduro Appiah,Instructional Faculty - Academic Year,NaN,642.00,0.0,3000.00,NaN,NaN,NaN,NaN,NaN,3642.00,205.44,3847.44,NaN,2024,California State University,NaN


In [15]:
def prepare_names(df):
    df = df.copy()
    tokens = df["name"].astype("string").str.casefold().str.split()

    df["_match_full"] = tokens.str.join(" ")
    df["_match_first"] = tokens.str[0]
    df["_match_last"] = tokens.str[-1]
    df["_match_count"] = tokens.str.len()

    # Exclude missing or empty names so they cannot match each other.
    return df.loc[df["_match_count"].gt(0)].copy()

pay = prepare_names(csu_pay)
faculty = prepare_names(combined)

# Find candidates with matching first and last tokens.
candidates = pay.merge(
    faculty,
    on=["_match_first", "_match_last"],
    how="inner",
    suffixes=("_transparentCA", "_scraped"),
)

# If token counts are equal, the full names must also match.
keep = (
    candidates["_match_full_transparentCA"].eq(candidates["_match_full_scraped"])
    | candidates["_match_count_transparentCA"].ne(candidates["_match_count_scraped"])
)

joined = (
    candidates.loc[keep]
    .drop(columns=[
        "_match_first",
        "_match_last",
        "_match_full_transparentCA",
        "_match_full_scraped",
        "_match_count_transparentCA",
        "_match_count_scraped",
    ])
    .reset_index(drop=True)
)

joined = joined.drop(columns=["name_transparentCA"]).rename(
    columns={"name_scraped": "name"}
)

joined = joined[["name"] + [col for col in joined.columns if col != "name"]]
joined.to_parquet("../Data/csu_joined_pay.parquet", index=False)

print(f"Matched rows: {len(joined):,}")
joined.head(10)

Matched rows: 15,501


,name,job,department_transparentCA,base,overtime,other,benefitsee,benefitser,benefitsdc,benefitshw,...,year_transparentCA,agency,status,title,department_scraped,school,year_scraped,major,source_url,start_date
0,Stephen Sillett,Instructional Faculty - 12 Month,NaN,147768.12,0.0,0.00,NaN,NaN,NaN,NaN,...,2024,California State University,NaN,Professor,"Forest, Fire, Range",Cal Poly Humboldt,NaN,<NA>,NaN,NaN
1,Arne Jacobson,Instructional Faculty - 12 Month,NaN,144585.88,0.0,0.00,NaN,NaN,NaN,NaN,...,2024,California State University,NaN,Professor,Engineering,Cal Poly Humboldt,NaN,<NA>,NaN,NaN
2,Arne Jacobson,Instructional Faculty - 12 Month,NaN,144585.88,0.0,0.00,NaN,NaN,NaN,NaN,...,2024,California State University,NaN,Director,Schatz Energy Resrch,Cal Poly Humboldt,NaN,<NA>,NaN,NaN
3,Rouhollah Aghasaleh,Instructional Faculty - Academic Year,NaN,88802.72,0.0,0.00,NaN,NaN,NaN,NaN,...,2024,California State University,NaN,Professor,"Education, School of",Cal Poly Humboldt,NaN,<NA>,NaN,NaN
4,Cinthya Ammerman,Instructional Faculty - Academic Year,NaN,77101.34,0.0,23194.75,NaN,NaN,NaN,NaN,...,2024,California State University,NaN,Assistant Professor,Native Amer Studies,Cal Poly Humboldt,NaN,<NA>,NaN,NaN
5,Debbie Gonzalez,Instructional Faculty - Academic Year,NaN,79388.39,0.0,35380.80,NaN,NaN,NaN,NaN,...,2024,California State University,NaN,Associate Professor,Social Work,Cal Poly Humboldt,NaN,<NA>,NaN,NaN
6,Daniela Cerbino,Instructional Faculty - Academic Year,NaN,75534.79,0.0,10949.19,NaN,NaN,NaN,NaN,...,2024,California State University,NaN,Assistant Professor,Applied Humanities,Cal Poly Humboldt,NaN,<NA>,NaN,NaN
7,Daniela Cerbino,Instructional Faculty - Academic Year,NaN,75534.79,0.0,10949.19,NaN,NaN,NaN,NaN,...,2024,California State University,NaN,Assistant Professor,English,Cal Poly Humboldt,NaN,<NA>,NaN,NaN
8,Bonnie Ludka,Instructional Faculty - Academic Year,NaN,90624.97,0.0,8248.74,NaN,NaN,NaN,NaN,...,2024,California State University,NaN,Assistant Professor,Engineering,Cal Poly Humboldt,NaN,<NA>,NaN,NaN
9,Bonnie Ludka,Instructional Faculty - Academic Year,NaN,90624.97,0.0,8248.74,NaN,NaN,NaN,NaN,...,2024,California State University,NaN,Lecturer,Math & Data Science,Cal Poly Humboldt,NaN,<NA>,NaN,NaN
